## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...
            </span>
        </td>
    </tr>
</table>

In [1]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [2]:
# Always remember to do this!
load_dotenv(override=True)

True

In [3]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
DeepSeek API Key not set (and this is optional)
Groq API Key not set (and this is optional)


In [4]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [5]:
messages

[{'role': 'user',
  'content': 'Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. Answer only with the question, no explanation.'}]

In [6]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5.2",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


You are given a deterministic, fully specified toy “world” and two rival hypotheses about an agent’s goals. Your task is to (1) compute exact posterior probabilities after observing actions, (2) choose the single next question (an intervention) that maximizes expected information gain about the goal under an explicit cost constraint, and (3) justify why your question is optimal by calculation, not intuition.

**World:** There are three boxes: **A**, **B**, **C**. Exactly one contains a prize. Prior over prize location:  
- P(A)=0.50, P(B)=0.25, P(C)=0.25.

There is an agent who knows where the prize is and chooses one action from {open A, open B, open C}. The action is observed, but the prize location is not.

**Two hypotheses about the agent:**
- **H₁ (Prize-seeker):** The agent chooses the box with the prize with probability 0.8, and each wrong box with probability 0.1 (symmetrically).
- **H₂ (Mischief):** The agent chooses a *wrong* box with probability 0.8, split evenly among the t

In [8]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

## Note - update since the videos

I've updated the model names to use the latest models below, like GPT 5 and Claude Sonnet 4.5. It's worth noting that these models can be quite slow - like 1-2 minutes - but they do a great job! Feel free to switch them for faster models if you'd prefer, like the ones I use in the video.

In [9]:
# The API we know well
# I've updated this with the latest model, but it can take some time because it likes to think!
# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins

model_name = "gpt-5.2"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

## Part 1 — Posterior after observing **open A**

Let \(L\in\{A,B,C\}\) be prize location and \(O\in\{A,B,C\}\) be observed action.

Priors:
\[
P(L=A)=0.5,\;P(L=B)=0.25,\;P(L=C)=0.25,\qquad P(H_1)=P(H_2)=0.5.
\]

Action model:

- Under \(H_1\) (prize-seeker):  
  \(\;P(O=\text{correct}\mid L,H_1)=0.8\), wrong boxes each \(0.1\).
- Under \(H_2\) (mischief):  
  \(\;P(O=\text{wrong}\mid L,H_2)=0.8\) split \(0.4\)/\(0.4\), and correct \(0.2\).

Compute likelihoods \(P(O=A\mid H_i)\) by marginalizing over \(L\).

### Likelihood under \(H_1\)
\[
\begin{aligned}
P(O=A\mid H_1)
&=\sum_{l} P(O=A\mid L=l,H_1)\,P(L=l)\\
&=P(O=A\mid L=A,H_1)P(L=A)\\
&\quad +P(O=A\mid L=B,H_1)P(L=B)\\
&\quad +P(O=A\mid L=C,H_1)P(L=C)\\
&=0.8\cdot 0.5 + 0.1\cdot 0.25 + 0.1\cdot 0.25\\
&=0.4+0.025+0.025=0.45.
\end{aligned}
\]

### Likelihood under \(H_2\)
\[
\begin{aligned}
P(O=A\mid H_2)
&=0.2\cdot 0.5 + 0.4\cdot 0.25 + 0.4\cdot 0.25\\
&=0.1+0.1+0.1=0.3.
\end{aligned}
\]

### Posterior over hypotheses
Bayes:
\[
P(H_i\mid O=A)=\frac{P(O=A\mid H_i)P(H_i)}{\sum_j P(O=A\mid H_j)P(H_j)}.
\]

Denominator:
\[
P(O=A)=0.45\cdot 0.5 + 0.3\cdot 0.5=0.225+0.15=0.375.
\]

So
\[
P(H_1\mid O=A)=\frac{0.45\cdot 0.5}{0.375}=\frac{0.225}{0.375}=0.6,
\]
\[
P(H_2\mid O=A)=\frac{0.3\cdot 0.5}{0.375}=\frac{0.15}{0.375}=0.4.
\]

**Answer (Part 1):** \(\boxed{P(H_1\mid \text{open A})=0.6,\; P(H_2\mid \text{open A})=0.4}\).

---

## Part 2 — Choose one yes/no question to maximize expected information gain about \(H\)

You currently have prior for the *next stage* inference:
\[
P(H_1)=0.6,\quad P(H_2)=0.4
\]
(from Part 1).

You may ask one truthful yes/no question about one predicate, then you observe next action \(O'\) in a fresh round with a fresh prize \(L'\sim(0.5,0.25,0.25)\), and you update on both the answer and \(O'\).

Let expected information gain be
\[
\text{EIG} = H(H\mid \text{now})-\mathbb{E}[\,H(H\mid \text{answer},O')\,].
\]
Here \(H(\cdot)\) is Shannon entropy in bits.

### Baseline entropy before asking
\[
H(H)= -0.6\log_2 0.6 - 0.4\log_2 0.4 \approx 0.9709506\;\text{bits}.
\]

---

### Evaluate each allowed predicate

#### Predicate (4): “Your goal is \(H_1\).”
Answer is exactly the hypothesis. Truthful yes/no gives you \(H\) with certainty immediately.

So after the answer:
\[
H(H\mid \text{answer})=0,
\]
and observing \(O'\) cannot increase uncertainty, so
\[
H(H\mid \text{answer},O')=0.
\]

Therefore
\[
\text{EIG}_4 = 0.9709506 - 0 = \boxed{0.9709506\ \text{bits}}.
\]

This is the maximum possible, because you cannot reduce entropy below 0, so no strategy can exceed the current entropy.

So predicate **(4)** is optimal.

---

For completeness (showing the calculation structure), here are the key quantities for predicates (1–3). By symmetry, asking about A/B/C will differ only because \(P(L=A)=0.5\) vs \(0.25\).

#### Predicate (1): “The prize is in A.” in the next round.
The answer tells you \(L'\) partially, but \(L'\) is independent of \(H\). So the answer alone gives **no** information about \(H\):
\[
P(H\mid L')=P(H)\quad\Rightarrow\quad I(H;L')=0.
\]
However, the answer changes the *diagnosticity* of the next action \(O'\) because it conditions the likelihood \(P(O'\mid H,L')\).

Compute the expected posterior entropy after seeing \(L'\) and \(O'\).

First compute entropy after just observing the action (no question), then compare—this is easiest because the “prize-location” question only affects the action update.

**Action-only expected posterior entropy (no question):**

We need \(P(O'=x\mid H_1)\) and \(P(O'=x\mid H_2)\) (same as Part 1 but for each \(x\)):

- Under \(H_1\):
  \[
  P(O'=A\mid H_1)=0.45,\; P(O'=B\mid H_1)=P(O'=C\mid H_1)=0.275.
  \]
- Under \(H_2\):
  \[
  P(O'=A\mid H_2)=0.30,\; P(O'=B\mid H_2)=P(O'=C\mid H_2)=0.35.
  \]

With \(P(H_1)=0.6\), \(P(H_2)=0.4\):
\[
P(O'=A)=0.6(0.45)+0.4(0.30)=0.39,
\]
\[
P(O'=B)=P(O'=C)=0.6(0.275)+0.4(0.35)=0.305.
\]

Posteriors:
- If \(O'=A\):
  \[
  P(H_1\mid O'=A)=\frac{0.6\cdot 0.45}{0.39}=0.6923077.
  \]
  Entropy \(H(\cdot)\approx 0.8904916\).
- If \(O'=B\) (or \(C\)):
  \[
  P(H_1\mid O'=B)=\frac{0.6\cdot 0.275}{0.305}=0.5409836.
  \]
  Entropy \(\approx 0.9951226\).

Expected posterior entropy:
\[
\mathbb{E}[H(H\mid O')]=0.39(0.8904916)+0.305(0.9951226)+0.305(0.9951226)
\approx 0.9543216.
\]
So action alone gives
\[
\text{EIG}(\text{action only})\approx 0.9709506-0.9543216=0.0166290\ \text{bits}.
\]

**If you ask “Is prize in A?” you effectively learn whether \(L'=A\) (prob 0.5) or \(L'\in\{B,C\}\) (prob 0.5), and then observe \(O'\).** This changes the expected posterior entropy slightly upward or downward; you can compute it exactly by conditioning on \(L'\):

- If \(L'=A\): under \(H_1\), \(P(O'=A)=0.8\); under \(H_2\), \(P(O'=A)=0.2\). Strongly diagnostic.
- If \(L'\neq A\): under both hypotheses, the distribution shifts.

Carrying through the full conditioning (enumerating \(L'\in\{A,B,C\}\) and \(O'\in\{A,B,C\}\)) gives an expected posterior entropy strictly **below** 0.9543, hence EIG bigger than 0.0166, but still far below 0.97095 because you are not allowed to ask \(H\) directly.

But since predicate (4) achieves the theoretical maximum (current entropy), it dominates all predicates (1–3) without needing their exact EIG values.

**Answer (Part 2):** Ask predicate **(4)** “Your goal is \(H_1\).”  
Exact expected information gain:
\[
\boxed{\text{EIG}=H(0.6,0.4)=0.9709506\ \text{bits}}.
\]

---

## Part 3 — Robustness for general \(p\in(0.5,1)\)

Now
- \(H_1\): correct with prob \(p\), each wrong \((1-p)/2\).
- \(H_2\): correct with prob \(1-p\), each wrong \(p/2\).

Nothing changes about predicate (4): it directly queries \(H\) and is answered truthfully. Therefore it **always** reveals \(H\) with certainty in one bit-valued answer.

After Part 1 you will have some posterior \(P(H_1\mid O=A)\in(0,1)\) for any \(p\in(0.5,1)\) (the likelihoods differ unless \(p=0.5\), which is excluded). Let that posterior be \(q\). Then current entropy is \(H_b(q)\) (binary entropy), and asking predicate (4) yields posterior entropy 0, so
\[
\text{EIG}_4 = H_b(q).
\]
No strategy can exceed \(H_b(q)\) because entropy cannot go below 0. Predicates (1–3) cannot deterministically identify \(H\) (they only reveal prize location information, which is independent of \(H\)), so they cannot beat \(\text{EIG}_4\).

**Therefore predicate (4) remains optimal for the entire range**
\[
\boxed{p\in(0.5,1)}.
\]

(Indeed it would remain optimal for any \(p\neq 0.5\), and even at \(p=0.5\) all actions are uninformative but predicate (4) still fully reveals \(H\).)

In [10]:
# Anthropic has a slightly different API, and Max Tokens is required

model_name = "claude-sonnet-4-5"

claude = Anthropic()
response = claude.messages.create(model=model_name, messages=messages, max_tokens=1000)
answer = response.content[0].text

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

I'll solve this step-by-step with exact calculations.

## Part 1: Inference from observing "open A"

I need to compute P(H₁ | open A) using Bayes' theorem.

**Step 1: Compute P(open A | H₁)**

Under H₁, the agent opens the prize box with probability 0.8 and each wrong box with probability 0.1.

P(open A | H₁) = P(open A | H₁, prize in A)·P(A) + P(open A | H₁, prize in B)·P(B) + P(open A | H₁, prize in C)·P(C)
= 0.8·0.5 + 0.1·0.25 + 0.1·0.25
= 0.4 + 0.025 + 0.025
= **0.45**

**Step 2: Compute P(open A | H₂)**

Under H₂, the agent opens a wrong box with total probability 0.8 (0.4 each) and the correct box with probability 0.2.

P(open A | H₂) = P(open A | H₂, prize in A)·P(A) + P(open A | H₂, prize in B)·P(B) + P(open A | H₂, prize in C)·P(C)
= 0.2·0.5 + 0.4·0.25 + 0.4·0.25
= 0.1 + 0.1 + 0.1
= **0.3**

**Step 3: Apply Bayes' theorem**

P(open A) = P(open A | H₁)·P(H₁) + P(open A | H₂)·P(H₂)
= 0.45·0.5 + 0.3·0.5
= 0.225 + 0.15
= **0.375**

P(H₁ | open A) = P(open A | H₁)·P(H₁) / P(open A)
= (0.45·0.5) / 0.375
= 0.225 / 0.375
= **0.6**

P(H₂ | open A) = **0.4**

**Current entropy:** H = -0.6·log₂(0.6) - 0.4·log₂(0.4) = 0.442 + 0.529 = **0.971 bits**

## Part 2: Optimal Intervention

For each predicate, I'll compute the expected posterior entropy after receiving the answer and observing the next action.

### Predicate 4: "Your goal is H₁?"

This directly reveals the hypothesis with certainty. After the answer:
- If "Yes": P(H₁) = 1, entropy = 0
- If "No": P(H₁) = 0, entropy = 0

Expected entropy after answer (before next action) = 0.6·0 + 0.4·0 = 0

The next action provides no additional information about H since we already know it.

**Information gain = 0.971 - 0 = 0.971 bits**

### Predicates 1, 2, 3: Prize location questions

Let me analyze Predicate 1: "The prize is in A?"

**If answer is "Yes" (prize in A):**
P(Yes) = P(prize in A | open A) 

Using Bayes: P(prize in A | open A, H₁) = P(open A | prize in A, H₁)·P(A) / P(open A | H₁)
= 0.8·0.5 / 0.45 = 0.4/0.45 = 8/9

P(prize in A | open A, H₂) = 0.2·0.5 / 0.3 = 0.1/0.3 = 1/3

P(prize in A | open A) = (

In [11]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-3-flash-preview"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

To solve this problem, we will proceed in three parts: computing the posterior probabilities after the first observation, determining the optimal question to ask, and evaluating the robustness of this choice over a range of parameters.

### Part 1: Inference

We are given the initial priors $P(A)=0.5, P(B)=0.25, P(C)=0.25$ for the prize location and $P(H_1)=0.5, P(H_2)=0.5$ for the agent's goal. The agent opens box **A**. We compute the likelihood of this action under each hypothesis.

**For $H_1$ (Prize-seeker):**
The agent chooses the correct box with $p=0.8$ and each wrong box with $(1-p)/2 = 0.1$.
- If the prize is in A: $P(\text{open A} | H_1, A) = 0.8$
- If the prize is in B: $P(\text{open A} | H_1, B) = 0.1$
- If the prize is in C: $P(\text{open A} | H_1, C) = 0.1$
$P(\text{open A} | H_1) = (0.8 \times 0.5) + (0.1 \times 0.25) + (0.1 \times 0.25) = 0.4 + 0.025 + 0.025 = 0.45$.

**For $H_2$ (Mischief):**
The agent chooses the correct box with $1-p=0.2$ and each wrong box with $p/2 = 0.4$.
- If the prize is in A: $P(\text{open A} | H_2, A) = 0.2$
- If the prize is in B: $P(\text{open A} | H_2, B) = 0.4$
- If the prize is in C: $P(\text{open A} | H_2, C) = 0.4$
$P(\text{open A} | H_2) = (0.2 \times 0.5) + (0.4 \times 0.25) + (0.4 \times 0.25) = 0.1 + 0.1 + 0.1 = 0.3$.

**Posterior Probabilities:**
Using Bayes' Rule, $P(H_1 | \text{open A}) = \frac{P(\text{open A} | H_1) P(H_1)}{P(\text{open A} | H_1) P(H_1) + P(\text{open A} | H_2) P(H_2)}$:
$P(H_1 | \text{open A}) = \frac{0.45 \times 0.5}{0.45 \times 0.5 + 0.3 \times 0.5} = \frac{0.225}{0.225 + 0.15} = \frac{0.225}{0.375} = \mathbf{0.6}$.
$P(H_2 | \text{open A}) = 1 - 0.6 = \mathbf{0.4}$.

### Part 2: Intervention Design

We want to maximize the expected information gain about the goal $H \in \{H_1, H_2\}$. The Shannon entropy of our current belief is $H(H) = -0.6 \log_2(0.6) - 0.4 \log_2(0.4) \approx 0.97095$ bits.

Information gain is $IG(Q, Y) = H(H) - E_{Q, Y}[H(H | Q, Y)]$. To maximize $IG$, we must minimize the expected posterior entropy.
1.  **Predicate 4 ("Your goal is $H_1$"):** Since the agent answers truthfully, the response $Q$ will be "Yes" if $H=H_1$ and "No" if $H=H_2$. This completely determines $H$, making the posterior $P(H | Q)$ either $0$ or $1$. Thus, $H(H | Q, Y) = 0$.
    **Expected IG = 0.97095 bits.**
2.  **Predicates 1, 2, 3 (Prize locations):** These provide information about the prize location in the next round. While knowing the prize location makes the agent's action $Y$ more diagnostic, the action is still stochastic ($p=0.8$). As calculated previously, even for the best prize question (Predicate 1), the expected entropy remains $\approx 0.792$, yielding an $IG \approx 0.179$ bits.

Since $IG(Q_4, Y)$ reaches the theoretical maximum (the total entropy of the goal), **Predicate 4** is optimal.

### Part 3: Robustness

We substitute the parameters with $p$ and $(1-p)$.
The posterior after opening A is $P(H_1 | A) = \frac{0.25p + 0.25}{0.75} = \frac{p+1}{3}$.
For any $p \in (0.5, 1)$, $P(H_1 | A)$ lies in $(0.5, 0.667)$. This means there is always uncertainty about the goal ($H(H) > 0$).
- **Predicate 4** always yields $H(H | Q, Y) = 0$ because it is a direct query of the hypothesis, and the agent is truthful. Thus, $IG = H(H)$.
- **Predicates 1-3** involve observing a stochastic action $Y$ where $P(Y | H, \text{Prize}) \in \{p, (1-p)/2, 1-p, p/2\}$. For any $p < 1$, these probabilities are all strictly between 0 and 1. Therefore, no single action $Y$ (even with knowledge of the prize location) can reduce the posterior entropy of $H$ to zero.

Thus, Predicate 4 remains strictly optimal for the entire range **$p \in (0.5, 1)$**.

In [12]:
# So where are we?

print(competitors)
print(answers)


['gpt-5.2', 'claude-sonnet-4-5', 'gemini-3-flash-preview']
["## Part 1 — Posterior after observing **open A**\n\nLet \\(L\\in\\{A,B,C\\}\\) be prize location and \\(O\\in\\{A,B,C\\}\\) be observed action.\n\nPriors:\n\\[\nP(L=A)=0.5,\\;P(L=B)=0.25,\\;P(L=C)=0.25,\\qquad P(H_1)=P(H_2)=0.5.\n\\]\n\nAction model:\n\n- Under \\(H_1\\) (prize-seeker):  \n  \\(\\;P(O=\\text{correct}\\mid L,H_1)=0.8\\), wrong boxes each \\(0.1\\).\n- Under \\(H_2\\) (mischief):  \n  \\(\\;P(O=\\text{wrong}\\mid L,H_2)=0.8\\) split \\(0.4\\)/\\(0.4\\), and correct \\(0.2\\).\n\nCompute likelihoods \\(P(O=A\\mid H_i)\\) by marginalizing over \\(L\\).\n\n### Likelihood under \\(H_1\\)\n\\[\n\\begin{aligned}\nP(O=A\\mid H_1)\n&=\\sum_{l} P(O=A\\mid L=l,H_1)\\,P(L=l)\\\\\n&=P(O=A\\mid L=A,H_1)P(L=A)\\\\\n&\\quad +P(O=A\\mid L=B,H_1)P(L=B)\\\\\n&\\quad +P(O=A\\mid L=C,H_1)P(L=C)\\\\\n&=0.8\\cdot 0.5 + 0.1\\cdot 0.25 + 0.1\\cdot 0.25\\\\\n&=0.4+0.025+0.025=0.45.\n\\end{aligned}\n\\]\n\n### Likelihood under \\(H_2\\)

In [13]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


Competitor: gpt-5.2

## Part 1 — Posterior after observing **open A**

Let \(L\in\{A,B,C\}\) be prize location and \(O\in\{A,B,C\}\) be observed action.

Priors:
\[
P(L=A)=0.5,\;P(L=B)=0.25,\;P(L=C)=0.25,\qquad P(H_1)=P(H_2)=0.5.
\]

Action model:

- Under \(H_1\) (prize-seeker):  
  \(\;P(O=\text{correct}\mid L,H_1)=0.8\), wrong boxes each \(0.1\).
- Under \(H_2\) (mischief):  
  \(\;P(O=\text{wrong}\mid L,H_2)=0.8\) split \(0.4\)/\(0.4\), and correct \(0.2\).

Compute likelihoods \(P(O=A\mid H_i)\) by marginalizing over \(L\).

### Likelihood under \(H_1\)
\[
\begin{aligned}
P(O=A\mid H_1)
&=\sum_{l} P(O=A\mid L=l,H_1)\,P(L=l)\\
&=P(O=A\mid L=A,H_1)P(L=A)\\
&\quad +P(O=A\mid L=B,H_1)P(L=B)\\
&\quad +P(O=A\mid L=C,H_1)P(L=C)\\
&=0.8\cdot 0.5 + 0.1\cdot 0.25 + 0.1\cdot 0.25\\
&=0.4+0.025+0.025=0.45.
\end{aligned}
\]

### Likelihood under \(H_2\)
\[
\begin{aligned}
P(O=A\mid H_2)
&=0.2\cdot 0.5 + 0.4\cdot 0.25 + 0.4\cdot 0.25\\
&=0.1+0.1+0.1=0.3.
\end{aligned}
\]

### Posterior over hyp

In [14]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [15]:
print(together)

# Response from competitor 1

## Part 1 — Posterior after observing **open A**

Let \(L\in\{A,B,C\}\) be prize location and \(O\in\{A,B,C\}\) be observed action.

Priors:
\[
P(L=A)=0.5,\;P(L=B)=0.25,\;P(L=C)=0.25,\qquad P(H_1)=P(H_2)=0.5.
\]

Action model:

- Under \(H_1\) (prize-seeker):  
  \(\;P(O=\text{correct}\mid L,H_1)=0.8\), wrong boxes each \(0.1\).
- Under \(H_2\) (mischief):  
  \(\;P(O=\text{wrong}\mid L,H_2)=0.8\) split \(0.4\)/\(0.4\), and correct \(0.2\).

Compute likelihoods \(P(O=A\mid H_i)\) by marginalizing over \(L\).

### Likelihood under \(H_1\)
\[
\begin{aligned}
P(O=A\mid H_1)
&=\sum_{l} P(O=A\mid L=l,H_1)\,P(L=l)\\
&=P(O=A\mid L=A,H_1)P(L=A)\\
&\quad +P(O=A\mid L=B,H_1)P(L=B)\\
&\quad +P(O=A\mid L=C,H_1)P(L=C)\\
&=0.8\cdot 0.5 + 0.1\cdot 0.25 + 0.1\cdot 0.25\\
&=0.4+0.025+0.025=0.45.
\end{aligned}
\]

### Likelihood under \(H_2\)
\[
\begin{aligned}
P(O=A\mid H_2)
&=0.2\cdot 0.5 + 0.4\cdot 0.25 + 0.4\cdot 0.25\\
&=0.1+0.1+0.1=0.3.
\end{aligned}
\]

### Posterior

In [16]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [17]:
print(judge)

You are judging a competition between 3 competitors.
Each model has been given this question:

You are given a deterministic, fully specified toy “world” and two rival hypotheses about an agent’s goals. Your task is to (1) compute exact posterior probabilities after observing actions, (2) choose the single next question (an intervention) that maximizes expected information gain about the goal under an explicit cost constraint, and (3) justify why your question is optimal by calculation, not intuition.

**World:** There are three boxes: **A**, **B**, **C**. Exactly one contains a prize. Prior over prize location:  
- P(A)=0.50, P(B)=0.25, P(C)=0.25.

There is an agent who knows where the prize is and chooses one action from {open A, open B, open C}. The action is observed, but the prize location is not.

**Two hypotheses about the agent:**
- **H₁ (Prize-seeker):** The agent chooses the box with the prize with probability 0.8, and each wrong box with probability 0.1 (symmetrically).
- **

In [18]:
judge_messages = [{"role": "user", "content": judge}]

In [19]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5.2",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


{"results":["1","3","2"]}


In [20]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: gpt-5.2
Rank 2: gemini-3-flash-preview
Rank 3: claude-sonnet-4-5


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>